In [1]:
import pandas as pd

filepath_jppost_str = '../../datasets/jppost/KEN_ALL.CSV'

# Read jppost csv
jppost_df = pd.read_csv(filepath_jppost_str, encoding='shift-jis', header=None)

# Select the prefecture column and drop duplicates
prefectures_sr = jppost_df[6]
prefectures_sr = prefectures_sr.drop_duplicates()
prefectures_list = prefectures_sr.tolist()

# Select the municipalities column and drop duplicates
municipalities_sr = jppost_df[7]
municipalities_sr = municipalities_sr.drop_duplicates()

print(len(prefectures_list)) # 47
print(len(municipalities_sr)) # 1894

47
1894


In [2]:
# Extract ○○市
shi_without_ku_sr = municipalities_sr[municipalities_sr.str.endswith('市')]

# Extract 東京23区
ku_of_tokyo_sr = municipalities_sr[municipalities_sr.str.contains('^(?!.*市).*区$')]

# Extract others
kuchoson_sr = municipalities_sr[~municipalities_sr.str.endswith('市')]
kuchoson_sr = kuchoson_sr[~kuchoson_sr.str.contains('^(?!.*市).*区$')]


print(len(shi_without_ku_sr)) # 770
print(len(ku_of_tokyo_sr)) # 23
print(len(kuchoson_sr)) # 1101

770
23
1101


In [3]:
# Extract 地方区
shi_and_ku_sr = kuchoson_sr[kuchoson_sr.str.endswith('区')]

# Extract 町村
all_choson_sr = kuchoson_sr[~kuchoson_sr.str.endswith('区')]


print(len(shi_and_ku_sr)) # 175
print(len(all_choson_sr)) # 926

175
926


In [4]:
# Divide 地方区 into 市 and 区
# len(shiku_sr[shiku_sr.str.contains('[市]{2,}')]) -> 0

shi_and_ku_df = shi_and_ku_sr.str.extract('(.+)市(.+)')
shi_and_ku_df[2] = '市'
shi_with_ku_sr = shi_and_ku_df[0].str.cat(shi_and_ku_df[2])

shi_with_ku_sr = shi_with_ku_sr.drop_duplicates()
ku_local_sr = shi_and_ku_df[1]


print(len(shi_with_ku_sr)) # 20
print(len(ku_local_sr)) # 175

20
175


In [5]:
# Extract 町村 without 郡
choson_not_under_gun_sr = all_choson_sr[~all_choson_sr.str.contains('郡')]
cho_not_under_gun_sr = choson_not_under_gun_sr[choson_not_under_gun_sr.str.endswith('町')]
son_not_under_gun_sr = choson_not_under_gun_sr[choson_not_under_gun_sr.str.endswith('村')]


print(cho_not_under_gun_sr) # 三宅島三宅村
print(son_not_under_gun_sr) # 八丈島八丈町

41081       大島町
41102    八丈島八丈町
Name: 7, dtype: object
41088       利島村
41089       新島村
41093      神津島村
41094    三宅島三宅村
41101      御蔵島村
41108      青ヶ島村
41109      小笠原村
Name: 7, dtype: object


In [6]:
# Extract 郡町村
gun_and_choson_sr = all_choson_sr[all_choson_sr.str.contains('郡')]


# Divide 郡町村 into 郡 and 町村
# len(gun_choson_sr[gun_choson_sr.str.contains('[郡]{2,}')]) -> 0

gun_and_choson_df = gun_and_choson_sr.str.extract('(.+)郡(.+)')
gun_and_choson_df[2] = '郡'
gun_sr = gun_and_choson_df[0].str.cat(gun_and_choson_df[2])
gun_sr = gun_sr.drop_duplicates()

choson_under_gun_sr = gun_and_choson_df[1]
cho_under_gun_sr = choson_under_gun_sr[choson_under_gun_sr.str.endswith('町')]
son_under_gun_sr = choson_under_gun_sr[choson_under_gun_sr.str.endswith('村')]


print(len(gun_sr)) # 361
print(len(cho_under_gun_sr)) # 741
print(len(son_under_gun_sr)) # 176

361
741
176


In [7]:
# Convert serieses to lists
municipalities_list = municipalities_sr.tolist()
shi_without_ku_list = shi_without_ku_sr.tolist()
ku_of_tokyo_list = ku_of_tokyo_sr.tolist()
shi_with_ku_list = shi_with_ku_sr.tolist()
ku_local_list = ku_local_sr.tolist()
cho_not_under_gun_list = cho_not_under_gun_sr.tolist()
son_not_under_gun_list = son_not_under_gun_sr.tolist()
gun_list = gun_sr.tolist()
cho_under_gun_list = cho_under_gun_sr.tolist()
son_under_gun_list = son_under_gun_sr.tolist()

# Combine shi lists and drop duplicates
shi_list = shi_without_ku_list + shi_with_ku_list
shi_list = list(set(shi_list))

# Combine ku lists and drop duplicates
ku_list = ku_of_tokyo_list + ku_local_list
ku_list = list(set(ku_list))

# Combine cho lists and drop duplicates
cho_list = cho_not_under_gun_list + cho_under_gun_list
cho_list = list(set(cho_list))

# Combine son lists and drop duplicates
son_list = son_not_under_gun_list + son_under_gun_list
son_list = list(set(son_list))


print(len(municipalities_list)) # 1894
print(len(shi_list)) # 790
print(len(ku_list)) # 132
print(len(gun_list)) # 361
print(len(cho_list)) # 717
print(len(son_list)) # 179

1894
790
132
361
717
179


In [8]:
filepath_output_base_str = '../../datasets/jppost/local_govt_lists/'
filepath_output_prefectures_str = filepath_output_base_str + '00_prefectures_list.txt'
filepath_output_shi_str = filepath_output_base_str + '01_shi_list.txt'
filepath_output_ku_str = filepath_output_base_str + '02_ku_list.txt'
filepath_output_gun_str = filepath_output_base_str + '03_gun_list.txt'
filepath_output_cho_str = filepath_output_base_str + '04_cho_list.txt'
filepath_output_son_str = filepath_output_base_str + '05_son_list.txt'
filepath_output_munucipalities_str = filepath_output_base_str + '99_municipalities_list.txt'


with open(filepath_output_prefectures_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in prefectures_list:
        fp.write(f'{item}\n')

with open(filepath_output_shi_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in shi_list:
        fp.write(f'{item}\n')

with open(filepath_output_ku_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in ku_list:
        fp.write(f'{item}\n')

with open(filepath_output_gun_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in gun_list:
        fp.write(f'{item}\n')

with open(filepath_output_cho_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in cho_list:
        fp.write(f'{item}\n')

with open(filepath_output_son_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in son_list:
        fp.write(f'{item}\n')

with open(filepath_output_munucipalities_str, mode='w', encoding='utf-8', newline='\n') as fp:
    for item in municipalities_list:
        fp.write(f'{item}\n')
